# Maximum Likelihood Estimation

Companion notebook for the [Maximum Likelihood Estimation](https://ml-viz.vercel.app/courses/probability-statistics/02-maximum-likelihood-estimation) lesson.

We'll visualize likelihood functions, derive MLE analytically, and connect MLE to cross-entropy loss.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Visualizing the likelihood function for Gaussian MLE

In [ ]:
rng = np.random.default_rng(42)
true_mu, true_sigma = 3.0, 1.5
data = rng.normal(true_mu, true_sigma, size=20)

# Log-likelihood as function of μ (with σ fixed at truth)
mu_range = np.linspace(0, 6, 300)
log_likelihoods = [stats.norm.logpdf(data, mu, true_sigma).sum() for mu in mu_range]

mle_mu = data.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(mu_range, log_likelihoods, color='#6366f1', lw=2)
ax.axvline(mle_mu, color='#f97316', lw=2, linestyle='--', label=f'MLE: μ̂ = {mle_mu:.3f}')
ax.axvline(true_mu, color='#2dd4bf', lw=2, linestyle=':', label=f'True μ = {true_mu}')
ax.set_xlabel('μ'); ax.set_ylabel('Log-likelihood')
ax.set_title('Log-likelihood ℓ(μ | data)'); ax.legend(); ax.grid(True, alpha=0.2)

ax2 = axes[1]
x_plot = np.linspace(-2, 8, 300)
ax2.hist(data, bins=8, density=True, alpha=0.6, color='#6366f1', label='Data')
ax2.plot(x_plot, stats.norm.pdf(x_plot, mle_mu, true_sigma),
         color='#f97316', lw=2, label=f'MLE fit N({mle_mu:.2f}, {true_sigma}²)')
ax2.plot(x_plot, stats.norm.pdf(x_plot, true_mu, true_sigma),
         color='#2dd4bf', lw=2, linestyle=':', label=f'True N({true_mu}, {true_sigma}²)')
ax2.set_title('MLE fit vs true distribution'); ax2.legend(); ax2.grid(True, alpha=0.2)

plt.tight_layout(); plt.show()

## MLE = minimizing cross-entropy

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

# Binary classification: one feature, logistic model
rng = np.random.default_rng(1)
n = 100
X = rng.normal(0, 1, n)
true_w, true_b = 2.0, -0.5
p_true = sigmoid(true_w * X + true_b)
y = rng.binomial(1, p_true)

def cross_entropy(params, X, y):
    w, b = params
    p = sigmoid(w * X + b)
    eps = 1e-12
    return -np.mean(y * np.log(p+eps) + (1-y) * np.log(1-p+eps))

result = optimize.minimize(cross_entropy, [0.0, 0.0], args=(X, y), method='BFGS')
w_mle, b_mle = result.x

print(f'True parameters:  w={true_w}, b={true_b}')
print(f'MLE parameters:   w={w_mle:.4f}, b={b_mle:.4f}')

fig, ax = plt.subplots(figsize=(9, 5))
x_plot = np.linspace(-3.5, 3.5, 300)
ax.scatter(X[y==0], y[y==0], color='#6366f1', alpha=0.5, s=30, label='y=0')
ax.scatter(X[y==1], y[y==1], color='#f97316', alpha=0.5, s=30, label='y=1')
ax.plot(x_plot, sigmoid(true_w*x_plot + true_b), color='#2dd4bf', lw=2, linestyle=':',
        label='True p(y=1|x)')
ax.plot(x_plot, sigmoid(w_mle*x_plot + b_mle), color='#f59e0b', lw=2.5,
        label='MLE fit')
ax.set_xlabel('x'); ax.set_ylabel('P(y=1 | x)')
ax.set_title('Logistic regression via MLE (= cross-entropy minimization)')
ax.legend(); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()